# 08 — Conformal Prediction Intervals

**What:** Wrap the hybrid model's point forecast in distribution-free
prediction intervals using split conformal prediction. For any significance
level $\alpha \in (0,1)$, the interval $[L_{t+1}, U_{t+1}]$ satisfies:

$$P\left(y_{t+1} \in [L_{t+1}, U_{t+1}]\right) \geq 1 - \alpha$$

This guarantee holds in finite samples without any assumption on the
data-generating process.

**Why:** The hybrid model minimises RMSE but is slightly miscalibrated
on QLIKE — it underestimates tail risk. A risk manager acts on the upper
bound of a plausible range, not a point forecast. The upper bound
$U_{t+1}$ is the operationally relevant quantity for margin requirements
and hedge ratios. Standard GARCH intervals are asymptotic and assume
correct model specification — both are debatable here. Conformal
prediction requires neither.

**How:** All conformal logic is encapsulated in `src/models/conformal.py`.
The validation set residuals from notebook 07 serve as the calibration set.
This notebook calls the class interface and interprets findings.

**Outline:**

0. Setup
1. Theory
2. Symmetric Calibration
3. Symmetric Visualisation
4. Symmetric Coverage Check
5. Why Asymmetric Intervals?
6. Asymmetric Calibration
7. Symmetric vs Asymmetric — Comparison
8. Asymmetric Coverage Check
9. Export
10. Conclusion

---
---
## 0. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.models.conformal import AsymmetricConformalPredictor, ConformalPredictor

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12, 'axes.labelsize': 11})

test_data = pd.read_csv(
    ROOT / 'data/processed/hybrid_test_predictions.csv',
    index_col='Date', parse_dates=True
)
val_data = pd.read_csv(
    ROOT / 'data/processed/hybrid_val_predictions.csv',
    index_col='Date', parse_dates=True
)

print(f'Calibration set : {len(val_data):,} rows | '
      f'{val_data.index[0].date()} → {val_data.index[-1].date()}')
print(f'Test set        : {len(test_data):,} rows | '
      f'{test_data.index[0].date()} → {test_data.index[-1].date()}')

---
---
## 1. Theory

**Split conformal prediction** computes nonconformity scores on a
held-out calibration set and uses their empirical quantile as the
interval half-width. For regression, the nonconformity score is:

$$s_i = |y_i - \hat{y}_i|, \quad i \in \mathcal{D}_{\text{cal}}$$

The prediction interval for a new point is:

$$C_\alpha = [\hat{y}_{T+1} - \hat{q},\; \hat{y}_{T+1} + \hat{q}]$$

where $\hat{q} = \text{Quantile}(\{s_i\}, 1-\alpha)$. The interval
width is constant across all test points — the method produces marginal,
not conditional, coverage.

**Why the validation set is the calibration set:** the validation set
was used only for XGBoost early stopping. The model never fitted to it,
so its residuals are exchangeable with the test residuals under
stationarity — the standard split conformal assumption.


---
---
## 2. Symmetric Calibration

In [ ]:
cp_sym = ConformalPredictor()
cp_sym.calibrate(
    y_true = val_data['y_true'].values,
    y_pred = val_data['y_pred_hybrid'].values,
)

print(cp_sym)
print()
for alpha in [0.20, 0.10, 0.05]:
    q = cp_sym.quantile(alpha)
    print(f'alpha={alpha:.2f} | coverage={1-alpha:.0%} | '
          f'half-width={q:.6f} | total width={2*q:.6f}')

---
---
## 3. Symmetric Visualisation

The hybrid point forecast with three nested symmetric conformal intervals at
80%, 90%, and 95% coverage. Both bounds are equidistant from the point
forecast — the interval half-width is the same on both sides.

In [ ]:
y_pred_test  = test_data['y_pred_hybrid'].values
y_true_test  = test_data['y_true'].values
test_index   = test_data.index

results_sym = cp_sym.predict_all(y_pred_test)

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(test_index,
                results_sym['0.05']['lower'], results_sym['0.05']['upper'],
                alpha=0.15, color='steelblue', label='95% interval')
ax.fill_between(test_index,
                results_sym['0.1']['lower'],  results_sym['0.1']['upper'],
                alpha=0.25, color='steelblue', label='90% interval')
ax.fill_between(test_index,
                results_sym['0.2']['lower'],  results_sym['0.2']['upper'],
                alpha=0.40, color='steelblue', label='80% interval')

ax.plot(test_index, y_pred_test, color='steelblue',
        linewidth=0.8, label='Hybrid forecast')
ax.plot(test_index, y_true_test, color='crimson',
        linewidth=0.6, alpha=0.8, label='Actual |r_{t+1}|')

ax.set_title('Symmetric Conformal Intervals — Equal strictness on both bounds')
ax.set_ylabel('Daily Absolute Return')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
---
## 4. Symmetric Coverage Check

Empirical coverage must be at least the nominal level for a valid
conformal predictor. A negative gap means the intervals are conservative.
A positive gap would indicate undercoverage — a violation of the guarantee.

In [ ]:
display(cp_sym.coverage_summary(y_true_test, y_pred_test))

---
---
## 5. Why Asymmetric Intervals?

The symmetric predictor above allocates the miscoverage budget equally
between the two sides of the interval:

$$\alpha_{\text{lower}} = \alpha_{\text{upper}} = \frac{\alpha}{2}$$

This is statistically clean but **operationally wrong** for volatility
forecasting. The cost function is not symmetric:

| Error direction | What happened | Consequence |
|---|---|---|
| $y_{t+1} > U_{t+1}$ | Model underestimated risk | Margin call, forced liquidation, capital breach |
| $y_{t+1} < L_{t+1}$ | Model overestimated risk | Excess capital held, opportunity cost |

Underestimating volatility — actual exceeds the upper bound — is far more
dangerous than overestimating it. A risk manager would rather hold too much
capital than face a margin call. The symmetric predictor ignores this
asymmetry entirely.

**The asymmetric solution** splits the $\alpha$ budget using a parameter
$\phi \in (0, 1)$ that controls how much of the miscoverage tolerance goes
to each tail:

$$\alpha_{\text{upper}} = \alpha \cdot \phi \qquad \alpha_{\text{lower}} = \alpha \cdot (1 - \phi)$$

With $\phi = 0.7$ (the value used here) and $\alpha = 0.10$:

$$\alpha_{\text{upper}} = 0.07 \quad \Rightarrow \quad \text{upper bound wrong only 7\% of the time (strict)}$$
$$\alpha_{\text{lower}} = 0.03 \quad \Rightarrow \quad \text{lower bound wrong only 3\% of the time (relaxed)}$$
$$\text{Total miscoverage} = 0.07 + 0.03 = 0.10 \quad \Rightarrow \quad 90\% \text{ coverage maintained} \checkmark$$

Instead of absolute residuals, the asymmetric predictor uses **signed residuals**:

$$r_i = y_{\text{true},i} - \hat{y}_i$$

A positive $r_i$ means the model underestimated (dangerous). A negative $r_i$
means the model overestimated (less costly). The interval is then:

$$q_{\text{lower}} = \text{Quantile}(\{r_i\},\; \alpha_{\text{lower}}) \qquad (\text{negative value})$$
$$q_{\text{upper}} = \text{Quantile}(\{r_i\},\; 1 - \alpha_{\text{upper}}) \qquad (\text{positive value})$$
$$\left[\ \hat{y}_{T+1} + q_{\text{lower}},\quad \hat{y}_{T+1} + q_{\text{upper}}\ \right]$$

The upper bound is **pushed further out** (more conservative) and the lower
bound is **pulled closer in** (less conservative) — redistributing coverage
toward the dangerous tail without changing the total guarantee.

---
---
## 6. Asymmetric Calibration

Calibrate the asymmetric predictor on the same validation set.
`asymmetry=0.7` allocates 70% of the $\alpha$ budget to the upper tail.

In [ ]:
PHI = 0.7  # fraction of alpha budget on upper tail

cp_asym = AsymmetricConformalPredictor(asymmetry=PHI)
cp_asym.calibrate(
    y_true = val_data['y_true'].values,
    y_pred = val_data['y_pred_hybrid'].values,
)

print(cp_asym)
print()
print(f'{"alpha":>6} | {"coverage":>9} | {"q_lower":>10} | {"q_upper":>10} | {"width":>10}')
print('-' * 56)
for alpha in [0.20, 0.10, 0.05]:
    q_lo = cp_asym.quantile_lower(alpha)
    q_hi = cp_asym.quantile_upper(alpha)
    q_sym = cp_sym.quantile(alpha)
    print(f'{alpha:>6.2f} | {1-alpha:>9.0%} | {q_lo:>10.6f} | {q_hi:>10.6f} | {q_hi-q_lo:>10.6f}'
          f'  (symmetric: ±{q_sym:.6f})')

---
---
## 7. Symmetric vs Asymmetric — Comparison

Both predictors at the 90% coverage level. The key difference is visible
in the **upper bound** — the asymmetric version pushes it higher,
providing a more conservative ceiling for risk management. The lower bound
moves closer to the forecast, reflecting that underestimation is less costly.

In [ ]:
results_asym = cp_asym.predict_all(y_pred_test)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# --- Symmetric (top panel) ---
ax1.fill_between(test_index,
                 results_sym['0.1']['lower'], results_sym['0.1']['upper'],
                 alpha=0.30, color='steelblue', label='90% symmetric interval')
ax1.plot(test_index, y_pred_test, color='steelblue', linewidth=0.8, label='Hybrid forecast')
ax1.plot(test_index, y_true_test, color='crimson',   linewidth=0.6, alpha=0.8, label='Actual |r|')
ax1.set_title(f'Symmetric  —  equal budget each side  '
              f'($\\alpha_{{lower}}$ = $\\alpha_{{upper}}$ = 5%)')
ax1.set_ylabel('Daily Absolute Return')
ax1.legend(loc='upper right')

# --- Asymmetric (bottom panel) ---
ax2.fill_between(test_index,
                 results_asym['0.1']['lower'], results_asym['0.1']['upper'],
                 alpha=0.30, color='darkorange', label='90% asymmetric interval  (φ=0.7)')
ax2.plot(test_index, y_pred_test, color='steelblue', linewidth=0.8, label='Hybrid forecast')
ax2.plot(test_index, y_true_test, color='crimson',   linewidth=0.6, alpha=0.8, label='Actual |r|')
ax2.set_title(f'Asymmetric  —  risk-oriented budget split  '
              f'($\\alpha_{{lower}}$ = 3%,  $\\alpha_{{upper}}$ = 7%)')
ax2.set_ylabel('Daily Absolute Return')
ax2.legend(loc='upper right')

fig.suptitle('Symmetric vs Asymmetric Conformal Intervals at 90% Coverage',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# quantitative comparison at 90%
q_sym  = cp_sym.quantile(0.10)
q_lo_a = cp_asym.quantile_lower(0.10)
q_hi_a = cp_asym.quantile_upper(0.10)

print(f'\n90% interval comparison:')
print(f'  Symmetric  : [{-q_sym:+.6f},  {+q_sym:+.6f}]  total width={2*q_sym:.6f}')
print(f'  Asymmetric : [{q_lo_a:+.6f},  {q_hi_a:+.6f}]  total width={q_hi_a-q_lo_a:.6f}')
print(f'  Upper bound shift : {q_hi_a - q_sym:+.6f}  (asymmetric upper is wider)')
print(f'  Lower bound shift : {abs(q_lo_a) - q_sym:+.6f}  (asymmetric lower is narrower)')

---
---
## 8. Asymmetric Coverage Check

The asymmetric predictor adds two columns to the coverage table:
**Lower Violations** (actual below lower bound) and **Upper Violations**
(actual above upper bound). With $\phi = 0.7$, we expect upper violations
to be roughly twice as frequent as lower violations — reflecting the
deliberate asymmetric allocation of the budget.

In [ ]:
display(cp_asym.coverage_summary(y_true_test, y_pred_test))

---
---
## 9. Export

Save the asymmetric predictor — this is the production model used by
the pipeline. The asymmetric intervals are exported as the canonical
output since they are operationally motivated for risk management.

In [ ]:
conformal_intervals = pd.DataFrame({
    'y_true'   : y_true_test,
    'y_pred'   : y_pred_test,
    'lower_80' : results_asym['0.2']['lower'],
    'upper_80' : results_asym['0.2']['upper'],
    'lower_90' : results_asym['0.1']['lower'],
    'upper_90' : results_asym['0.1']['upper'],
    'lower_95' : results_asym['0.05']['lower'],
    'upper_95' : results_asym['0.05']['upper'],
}, index=test_index)

conformal_intervals.to_csv(ROOT / 'data/processed/conformal_intervals.csv')
print(f'Intervals saved → {len(conformal_intervals):,} rows | '
      f'{conformal_intervals.index[0].date()} → '
      f'{conformal_intervals.index[-1].date()}')

cp_asym.save(ROOT / 'data/processed/conformal.pkl')
display(conformal_intervals.head())

---
---
## 10. Conclusion

This notebook built and compared two conformal predictors calibrated on
987 validation observations (2018–2022) and evaluated on 987 test
observations (2022–2026).

**Symmetric predictor.** The baseline. All three coverage levels are
met or exceeded — 80.67% at nominal 80%, 92.81% at nominal 90%, and
95.95% at nominal 95%. The conformal guarantee holds, confirming that
validation residuals are exchangeable with test residuals. However, the
intervals treat upper and lower errors as equally costly, which is
inconsistent with how volatility risk is managed in practice.

**Why symmetric is insufficient.** Underestimating volatility — actual
exceeds the upper bound — triggers margin calls, forced liquidations,
and capital breaches. Overestimating it merely results in excess capital
being held. The cost function is fundamentally asymmetric, and the
predictor should reflect that.

**Asymmetric predictor ($\phi = 0.7$).** The same total coverage guarantee
is maintained, but the $\alpha$ budget is redistributed: 70% to the upper
tail, 30% to the lower tail. At 90% nominal coverage, this means the upper
bound is allowed to fail only 7% of the time, while the lower bound can
fail 3% of the time. The upper bound is wider than the symmetric version,
the lower bound is narrower — exactly what a risk manager wants.

**Operational interpretation.** The upper bound $U_{t+1}$ of the
asymmetric 90% interval is the conservative ceiling a risk manager uses
to size positions. It will be exceeded on fewer days than the symmetric
equivalent, at the cost of holding slightly more capital on average.
This is the correct trade-off for a volatility risk management tool.

**What comes next.** Notebook 09 formally backtests these intervals
using the Kupiec Proportion of Failures (POF) test and validates the
asymmetric violation split against the nominal $\alpha_{\text{upper}}$
and $\alpha_{\text{lower}}$ targets.